# Classificação de gêneros com Word2Vec

## Representação utilizada: Word2Vec

Este notebook usa a mesma abordagem do `comparacao_modelos.ipynb` do repositório: treina `Word2Vec` com `vector_size=100`, `window=5`, `min_count=2`, `workers=1`, `sg=1` e `epochs=20`. O texto é separado em tokens com `.split()`, como no código de referência.

Como cada palavra tem um vetor e o classificador precisa de um vetor por livro, a função `embedding_livro` calcula a média dos vetores das palavras conhecidas. Se nenhum token estiver no vocabulário, o livro recebe um vetor de zeros. Diferente do notebook original, o Word2Vec é treinado apenas com os textos de treino, para não usar os textos de teste durante o ajuste da representação.

---

## 1. Bibliotecas e configuração

In [1]:
# Dependências principais (execute no terminal, se ainda não estiverem instaladas):
# pip install pandas numpy scikit-learn
# Para Word2Vec: pip install gensim
# Para SentenceTransformer: pip install sentence-transformers

from IPython.display import display


## 2. Carregamento, preparação do texto e criação dos rótulos

O código usa `text_final` sempre que essa coluna existe, como no notebook de referência. Para versões do CSV anteriores ao pré-processamento, usa `summary` ou `trecho_1_clean`. O campo `subjects` é convertido em classes de gênero por regras simples, pois o CSV do projeto não garante uma coluna `genre` pronta.

In [2]:
import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

CAMINHO_CSV = "gutenberg_books.csv"
SEMENTE = 42
TEST_SIZE = 0.25
MIN_AMOSTRAS_POR_GENERO = 4


def normalizar_para_regra(texto):
    """Normaliza o texto de subjects para facilitar a identificação por palavras-chave."""
    texto = "" if pd.isna(texto) else str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))


def identificar_genero(subjects):
    """Cria um rótulo aproximado usando os metadados subjects do Gutenberg.

    Os rótulos são fracos (derivados de regras), não anotações humanas revisadas.
    Se um livro mencionar mais de um gênero, a primeira regra correspondente prevalece.
    """
    s = normalizar_para_regra(subjects)
    regras = [
        ("Ficção científica/fantasia", [r"science fiction", r"fantasy fiction", r"fantasy stories", r"utopias?", r"dystopias?"]),
        ("Mistério/policial", [r"detective and mystery stories", r"detective stories", r"mystery fiction", r"crime fiction", r"police stories"]),
        ("Terror/gótico", [r"horror fiction", r"ghost stories", r"gothic fiction", r"supernatural fiction", r"vampire fiction"]),
        ("Romance", [r"love stories", r"romance fiction", r"romantic fiction", r"courtship.{0,30}fiction"]),
        ("Guerra", [r"war stories", r"war fiction", r"military fiction", r"world war.{0,30}fiction"]),
        ("Aventura", [r"adventure stories", r"adventure fiction", r"sea stories", r"pirates.{0,30}fiction"]),
        ("Ficção histórica", [r"historical fiction", r"historical stories"]),
        ("Faroeste", [r"western stories", r"cowboy stories", r"western fiction"]),
    ]
    for genero, padroes in regras:
        if any(re.search(padrao, s) for padrao in padroes):
            return genero
    return np.nan


df_original = pd.read_csv(CAMINHO_CSV, dtype={"book_id": str})
print(f"Livros carregados: {len(df_original)}")
print("Colunas disponíveis:", list(df_original.columns))

# O notebook de referência usa text_final como entrada. Se a coluna não existir,
# usamos o resumo original para manter o notebook executável com versões anteriores do CSV.
if "text_final" in df_original.columns:
    coluna_texto = "text_final"
elif "summary" in df_original.columns:
    coluna_texto = "summary"
elif "trecho_1_clean" in df_original.columns:
    coluna_texto = "trecho_1_clean"
else:
    raise ValueError(
        "Não encontrei 'text_final', 'summary' nem 'trecho_1_clean' no CSV. "
        "Confira o arquivo gutenberg_books.csv."
    )

df = df_original.copy()
df["texto"] = df[coluna_texto].fillna("").astype(str).str.strip()

# Se o CSV já tiver gêneros revisados manualmente, use-os. Caso contrário,
# derive rótulos aproximados a partir de subjects.
if "genero" in df.columns and df["genero"].notna().any():
    df["genero"] = df["genero"].astype(str).str.strip()
    df.loc[df["genero"].str.lower().isin(["", "nan", "none", "null"]), "genero"] = np.nan
else:
    if "subjects" not in df.columns:
        raise ValueError(
            "O CSV não tem gêneros anotados na coluna 'genero' nem metadados 'subjects'. "
            "Adicione uma coluna 'genero' manualmente ou recupere 'subjects' no scraping."
        )
    df["genero"] = df["subjects"].apply(identificar_genero)

df = df[(df["texto"] != "") & df["genero"].notna()].copy()

# Remove classes com poucos exemplos, para que seja possível dividir entre treino e teste.
contagem = df["genero"].value_counts()
generos_validos = contagem[contagem >= MIN_AMOSTRAS_POR_GENERO].index
df = df[df["genero"].isin(generos_validos)].reset_index(drop=True)

print(f"Coluna textual usada: {coluna_texto}")
print(f"Livros com texto e rótulo identificável: {len(df)}")
print("Distribuição dos gêneros aproximados:")
print(df["genero"].value_counts())

if df["genero"].nunique() < 2:
    raise ValueError(
        "Foram encontrados menos de dois gêneros com pelo menos "
        f"{MIN_AMOSTRAS_POR_GENERO} livros cada. Verifique a distribuição de 'subjects' "
        "ou crie/adicione uma coluna 'genero' manualmente. As regras são heurísticas e "
        "podem precisar ser adaptadas aos valores reais de subjects."
    )

X_texto = df["texto"].to_numpy()
y = df["genero"].to_numpy()
indices = np.arange(len(df))
idx_treino, idx_teste = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=SEMENTE,
    stratify=y,
)

texto_treino = df.iloc[idx_treino]["texto"].tolist()
texto_teste = df.iloc[idx_teste]["texto"].tolist()
y_treino = df.iloc[idx_treino]["genero"].to_numpy()
y_teste = df.iloc[idx_teste]["genero"].to_numpy()
print(f"Exemplos para treino: {len(texto_treino)} | para teste: {len(texto_teste)}")


Livros carregados: 312
Colunas disponíveis: ['book_id', 'title', 'author', 'language', 'loc_class', 'subjects', 'release_date', 'ebook_url', 'summary', 'work_key', 'trecho', 'summary_normalized', 'summary_clean', 'tokens', 'tokens_no_punctuation', 'tokens_final', 'text_final', 'tokens_lemmatized', 'text_lemmatized', 'tokens_stemmed', 'text_stemmed']
Coluna textual usada: text_final
Livros com texto e rótulo identificável: 151
Distribuição dos gêneros aproximados:
genero
Mistério/policial             45
Romance                       35
Ficção científica/fantasia    31
Aventura                      17
Terror/gótico                 10
Guerra                         8
Ficção histórica               5
Name: count, dtype: int64
Exemplos para treino: 113 | para teste: 38


## 3. Aplicação da representação

In [3]:
from gensim.models import Word2Vec

# Mesmos hiperparâmetros e estratégia de média dos vetores do notebook de referência.
tokens_treino = [texto.lower().split() for texto in texto_treino]
tokens_teste = [texto.lower().split() for texto in texto_teste]

modelo_w2v = Word2Vec(
    sentences=tokens_treino,
    vector_size=100,
    window=5,
    min_count=2,
    workers=1,
    sg=1,
    epochs=20,
)

def embedding_livro(tokens, modelo):
    vetores = [modelo.wv[token] for token in tokens if token in modelo.wv]
    if not vetores:
        return np.zeros(modelo.vector_size)
    return np.mean(vetores, axis=0)

X_treino = np.array([embedding_livro(tokens, modelo_w2v) for tokens in tokens_treino])
X_teste = np.array([embedding_livro(tokens, modelo_w2v) for tokens in tokens_teste])

print("Dimensão dos embeddings de treino:", X_treino.shape)
print("Dimensão dos embeddings de teste:", X_teste.shape)
print("Tamanho do vocabulário Word2Vec:", len(modelo_w2v.wv))


Dimensão dos embeddings de treino: (113, 100)
Dimensão dos embeddings de teste: (38, 100)
Tamanho do vocabulário Word2Vec: 1486


## 4. Treinamento e avaliação

Todos os notebooks usam a mesma divisão treino/teste e o mesmo classificador (`LogisticRegression`) para facilitar a comparação. As métricas são calculadas apenas no conjunto de teste.

In [4]:
# Um classificador Logistic Regression é usado em todos os notebooks para que a
# principal diferença entre os experimentos seja a representação textual.
classificador = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEMENTE)
classificador.fit(X_treino, y_treino)
predicoes = classificador.predict(X_teste)

print(f"Acurácia: {accuracy_score(y_teste, predicoes):.3f}")
print(f"F1 macro: {f1_score(y_teste, predicoes, average='macro', zero_division=0):.3f}")
print("\nRelatório de classificação:")
print(classification_report(y_teste, predicoes, zero_division=0))

rotulos = sorted(set(y_teste) | set(predicoes))
matriz_confusao = confusion_matrix(y_teste, predicoes, labels=rotulos)
display(pd.DataFrame(matriz_confusao, index=rotulos, columns=rotulos).rename_axis("Real \\ Previsto"))


Acurácia: 0.158
F1 macro: 0.112

Relatório de classificação:
                            precision    recall  f1-score   support

                  Aventura       0.12      0.25      0.17         4
Ficção científica/fantasia       0.00      0.00      0.00         8
          Ficção histórica       0.00      0.00      0.00         1
                    Guerra       0.00      0.00      0.00         2
         Mistério/policial       0.50      0.27      0.35        11
                   Romance       0.00      0.00      0.00         9
             Terror/gótico       0.17      0.67      0.27         3

                  accuracy                           0.16        38
                 macro avg       0.11      0.17      0.11        38
              weighted avg       0.17      0.16      0.14        38



,Aventura,Ficção científica/fantasia,Ficção histórica,Guerra,Mistério/policial,Romance,Terror/gótico
Real \ Previsto,,,,,,,
Aventura,1,0,0,0,1,0,2
Ficção científica/fantasia,2,0,0,1,1,0,4
Ficção histórica,0,0,0,0,0,0,1
Guerra,1,0,0,0,1,0,0
Mistério/policial,1,0,3,2,3,0,2
Romance,3,0,2,3,0,0,1
Terror/gótico,0,0,0,1,0,0,2


## 5. Limitações para interpretar o resultado

- Os rótulos são **aproximados**, gerados a partir de palavras-chave em `subjects`; não equivalem a uma anotação humana revisada.
- As regras atribuem apenas um gênero por livro e descartam exemplos sem gênero detectável. Verifique a distribuição impressa na seção 2.
- Se não houver ao menos duas classes com quatro livros cada, ajuste `identificar_genero()` à forma real dos metadados ou crie uma coluna `genero` com rótulos revisados.
- Como a base é pequena, os resultados podem variar bastante de acordo com os livros disponíveis e a divisão treino/teste.